## Multilayer perceptron

### data preparation

In [19]:
import pandas as pd
import numpy as np
import keras
import tensorflow as tf


data_dir = "/Users/matthew/Documents/Rutgers/25 fall/ML/ml_project/data/final_merged_clean.parquet"
data = pd.read_parquet(data_dir)

In [20]:
data.head()

,bond_cusip,ytm,date,rating_A,rating_AA,rating_AAA,rating_B,rating_BB,rating_BBB,rating_C,...,revtq_growth,niq_growth,sp500_ret,ir3m_chg,ir10y_chg,vix_chg,gdp_gr,cpi_infl,etf_price,etf_return
1,000361AB1,0.04827,2002-08-31,-1,-1,-1,-1,-1,-1,-1,...,-0.376664,0.382235,-0.079004,0.006,-0.359,0.261024,0.004065,0.002227,0.489120,0.508042
2,000361AB1,0.04386,2002-09-30,-1,-1,-1,-1,-1,-1,-1,...,-0.293506,0.738528,0.004881,-0.020,-0.328,0.019045,0.000000,0.002778,0.602528,-0.859122
3,000361AB1,0.04122,2002-10-31,-1,-1,-1,-1,-1,-1,-1,...,0.427600,0.852125,-0.110024,-0.118,-0.530,0.215993,0.000000,0.001662,0.495813,-0.576790
4,000361AB1,0.03873,2002-11-30,-1,-1,-1,-1,-1,-1,-1,...,0.444784,0.839266,0.086449,-0.110,0.304,-0.215419,0.001236,0.002212,0.495035,-0.418532
5,000361AB1,0.04015,2002-12-31,-1,-1,-1,-1,-1,-1,-1,...,0.276607,-0.706236,0.057070,-0.215,0.302,-0.116891,0.000000,0.001656,0.609422,0.652731


In [21]:
data.tail()

,bond_cusip,ytm,date,rating_A,rating_AA,rating_AAA,rating_B,rating_BB,rating_BBB,rating_C,...,revtq_growth,niq_growth,sp500_ret,ir3m_chg,ir10y_chg,vix_chg,gdp_gr,cpi_infl,etf_price,etf_return
886952,G9460GAA9,0.08405,2022-12-31,-1,-1,-1,-1,-1,-1,-1,...,0.578831,-0.544987,0.053753,0.278,-0.374,-0.204791,0.000000,0.002446,0.068211,-0.923228
886953,G9460GAA9,0.08608,2023-01-31,-1,-1,-1,-1,-1,-1,-1,...,-0.375252,-0.527276,-0.058971,0.007,0.176,0.052964,0.000000,0.000335,-0.052254,0.163472
886954,G9460GAA9,0.08248,2023-02-28,-1,-1,-1,-1,-1,-1,-1,...,-0.368442,-0.534020,0.061753,0.310,-0.350,-0.104753,0.007239,0.005515,0.071767,-0.427458
886955,G9460GAA9,0.07865,2023-03-31,-1,-1,-1,-1,-1,-1,-1,...,-0.340438,-0.539235,-0.026112,0.140,0.387,0.067010,0.000000,0.003395,-0.170339,-0.941419
886956,G9460GAA9,0.07835,2023-04-30,-1,-1,-1,-1,-1,-1,-1,...,0.210082,0.700000,0.035052,-0.125,-0.422,-0.096618,0.000000,0.000554,-0.226018,-0.572624


In [22]:
data.columns

Index(['bond_cusip', 'ytm', 'date', 'rating_A', 'rating_AA', 'rating_AAA',
       'rating_B', 'rating_BB', 'rating_BBB', 'rating_C', 'rating_CC',
       'rating_CCC', 'rating_D', 'upgrade', 'downgrade', 'costat_bin', 'tmt',
       'coupon', 't_spread', 'bond_ret', 'gs3m', 'term_spread', 'ret', 'vol',
       'dvol', 'turnover', 'mktcap', 'bidask', 'numtrades', 'price_mean',
       'log_atq', 'lev_total', 'equity_ratio', 'roa', 'profit_margin',
       'int_coverage', 'accounting_mktcap', 'market_to_book', 'atq_growth',
       'revtq_growth', 'niq_growth', 'sp500_ret', 'ir3m_chg', 'ir10y_chg',
       'vix_chg', 'gdp_gr', 'cpi_infl', 'etf_price', 'etf_return'],
      dtype='object')

In [23]:
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["bond_cusip", "date"])
data["ytm_change"] = data.groupby("bond_cusip")["ytm"].diff()
data = data.dropna(subset=["ytm_change"])


In [24]:
exclude_cols = [
    'date','cusip_x','company_symbol','issuer6','PERMNO','GVKEY','cusip_y',
    'rating_AA','rating_BBB','rating_B', 'rating_C', 'rating_CC',
     'rating_D', 'rating_CCC','rating_A','rating_AAA', 'rating_BB', 'gsector', 'month_y',
    'upgrade','downgrade','month_x','ytm_change'
]

numeric_cols = [
    col for col in data.columns
    if col not in exclude_cols and data[col].dtype != 'object'
]

print(f"We standarlize these lines:\n{numeric_cols}")

We standarlize these lines:
['ytm', 'costat_bin', 'tmt', 'coupon', 't_spread', 'bond_ret', 'gs3m', 'term_spread', 'ret', 'vol', 'dvol', 'turnover', 'mktcap', 'bidask', 'numtrades', 'price_mean', 'log_atq', 'lev_total', 'equity_ratio', 'roa', 'profit_margin', 'int_coverage', 'accounting_mktcap', 'market_to_book', 'atq_growth', 'revtq_growth', 'niq_growth', 'sp500_ret', 'ir3m_chg', 'ir10y_chg', 'vix_chg', 'gdp_gr', 'cpi_infl', 'etf_price', 'etf_return']


## Regression

In [25]:
import numpy as np
import keras
import pandas as pd
from itertools import product
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
)

def build_mlp_regressor(
    input_dim: int,
    n_hidden_layers: int = 3,
    hidden_units: int = 16,
    learning_rate: float = 1e-4,
) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name="features")
    h = inputs
    for _ in range(n_hidden_layers):
        h = keras.layers.Dense(hidden_units, activation="relu")(h)

    outputs = keras.layers.Dense(1, activation="linear", name="pred")(h)

    model = keras.Model(inputs=inputs, outputs=outputs)

    opt = keras.optimizers.Adam(
        learning_rate=learning_rate,
        clipnorm=1.0,
    )

    model.compile(
        loss="mse",
        optimizer=opt,
        metrics=["mae", "mse"],
    )
    return model

#  2. data

data_norm = data.sort_values("date").copy()

y_col = "ytm_change"
x_cols = [c for c in numeric_cols if c != y_col]
idx_cols = ["bond_cusip", "date"]

start_year = 2002
end_year   = 2023
train_year = 5
val_year   = 1
test_year  = 1

n_hidden_layers_list = [2, 3]
hidden_units_list    = [16, 32]
learning_rate_list   = [1e-3, 3e-4]
batch_size_list      = [128, 256]

config_list = list(product(
    n_hidden_layers_list,
    hidden_units_list,
    learning_rate_list,
    batch_size_list,
))

hp_scores = {cfg: [] for cfg in config_list}

print("= PHASE 1: tuning hyper-parameters on each rolling window =\n")

for start in range(start_year,
                   end_year - (train_year + val_year + test_year) + 1):
    train_start_year = start
    train_end_year   = start + train_year + val_year - 1
    test_start_year  = train_end_year + 1
    test_end_year    = train_end_year + test_year

    print(f"[TUNING] Rolling window: train {train_start_year}-{train_end_year}, "
          f"test {test_start_year}-{test_end_year}")

    m_train = ((data_norm["date"].dt.year >= train_start_year) &
               (data_norm["date"].dt.year <= train_end_year))
    m_test  = ((data_norm["date"].dt.year >= test_start_year) &
               (data_norm["date"].dt.year <= test_end_year))

    df_train = data_norm.loc[m_train].copy()
    df_test  = data_norm.loc[m_test].copy()

    if df_train.empty or df_test.empty:
        print("  -> skip (no data for this window)")
        continue

    X_train = df_train[x_cols].to_numpy(dtype="float32")
    y_train = df_train[y_col].to_numpy(dtype="float32")
    X_test  = df_test[x_cols].to_numpy(dtype="float32")
    y_test  = df_test[y_col].to_numpy(dtype="float32")

    for cfg in config_list:
        n_layers, units, lr, bs = cfg

        model = build_mlp_regressor(
            input_dim=X_train.shape[1],
            n_hidden_layers=n_layers,
            hidden_units=units,
            learning_rate=lr,
        )

        early_cb = keras.callbacks.EarlyStopping(
            patience=5,
            restore_best_weights=True,
            monitor="val_loss"
        )

        model.fit(
            X_train,
            y_train,
            batch_size=bs,
            epochs=50,
            validation_split=0.2,  
            shuffle=False,
            callbacks=[early_cb],
            verbose=0,
        )

        y_pred = model.predict(X_test, verbose=0).ravel()
        r2     = r2_score(y_test, y_pred)

        hp_scores[cfg].append(r2)

# R2 of each hyper-parameter pairs
avg_scores = {}
for cfg, scores in hp_scores.items():
    if len(scores) == 0:
        avg_scores[cfg] = -np.inf
    else:
        avg_scores[cfg] = float(np.mean(scores))

best_cfg = max(avg_scores, key=avg_scores.get)
best_score = avg_scores[best_cfg]

print("\n= BEST HYPER-PARAMETERS (based on average R2 across windows) =")
print("Best config (n_layers, units, lr, batch_size):", best_cfg)
print("Average validation R2 across windows:", best_score)

best_n_layers, best_units, best_lr, best_bs = best_cfg


print("\n= PHASE 2: final rolling-window run with best hyper-parameters =\n")

mlp_metrics_list = []
mlp_results_list = []

for start in range(start_year,
                   end_year - (train_year + val_year + test_year) + 1):
    train_start_year = start
    train_end_year   = start + train_year + val_year - 1
    test_start_year  = train_end_year + 1
    test_end_year    = train_end_year + test_year

    print(f"[FINAL] Rolling window: train {train_start_year}-{train_end_year}, "
          f"test {test_start_year}-{test_end_year}")

    m_train = ((data_norm["date"].dt.year >= train_start_year) &
               (data_norm["date"].dt.year <= train_end_year))
    m_test  = ((data_norm["date"].dt.year >= test_start_year) &
               (data_norm["date"].dt.year <= test_end_year))

    df_train = data_norm.loc[m_train].copy()
    df_test  = data_norm.loc[m_test].copy()

    if df_train.empty or df_test.empty:
        print("  -> skip (no data for this window)")
        continue

    X_train = df_train[x_cols].to_numpy(dtype="float32")
    y_train = df_train[y_col].to_numpy(dtype="float32")
    X_test  = df_test[x_cols].to_numpy(dtype="float32")
    y_test  = df_test[y_col].to_numpy(dtype="float32")

    model = build_mlp_regressor(
        input_dim=X_train.shape[1],
        n_hidden_layers=best_n_layers,
        hidden_units=best_units,
        learning_rate=best_lr,
    )

    early_cb = keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True,
        monitor="val_loss"
    )

    model.fit(
        X_train,
        y_train,
        batch_size=best_bs,
        epochs=50,
        validation_split=0.2,
        shuffle=False,
        callbacks=[early_cb],
        verbose=0,
    )

    y_pred = model.predict(X_test, verbose=0).ravel()

    mse   = mean_squared_error(y_test, y_pred)
    mae   = mean_absolute_error(y_test, y_pred)
    medae = median_absolute_error(y_test, y_pred)
    r2    = r2_score(y_test, y_pred)

    mlp_metrics_list.append({
        "train_start": train_start_year,
        "train_end":   train_end_year,
        "test_start":  test_start_year,
        "test_end":    test_end_year,
        "MSE": mse,
        "MAE": mae,
        "MedAE": medae,
        "R2": r2,
    })

    df_res = df_test[idx_cols].copy()
    df_res["model"] = "MLP"
    df_res["pred"]  = y_pred
    df_res["act"]   = y_test
    mlp_results_list.append(df_res)

df_mlp_metrics = pd.DataFrame(mlp_metrics_list)
df_mlp_results = pd.concat(mlp_results_list, ignore_index=True)

print("\n= Final rolling-window metrics (with tuned hyper-parameters) =")
print(df_mlp_metrics)

print("\nRolling-window average performance of tuned MLP:")
print("MSE   :", df_mlp_metrics["MSE"].mean())
print("MAE   :", df_mlp_metrics["MAE"].mean())
print("MedAE :", df_mlp_metrics["MedAE"].mean())
print("R2    :", df_mlp_metrics["R2"].mean())

= PHASE 1: tuning hyper-parameters on each rolling window =

[TUNING] Rolling window: train 2002-2007, test 2008-2008
[TUNING] Rolling window: train 2003-2008, test 2009-2009
[TUNING] Rolling window: train 2004-2009, test 2010-2010
[TUNING] Rolling window: train 2005-2010, test 2011-2011
[TUNING] Rolling window: train 2006-2011, test 2012-2012
[TUNING] Rolling window: train 2007-2012, test 2013-2013
[TUNING] Rolling window: train 2008-2013, test 2014-2014
[TUNING] Rolling window: train 2009-2014, test 2015-2015
[TUNING] Rolling window: train 2010-2015, test 2016-2016
[TUNING] Rolling window: train 2011-2016, test 2017-2017
[TUNING] Rolling window: train 2012-2017, test 2018-2018
[TUNING] Rolling window: train 2013-2018, test 2019-2019
[TUNING] Rolling window: train 2014-2019, test 2020-2020
[TUNING] Rolling window: train 2015-2020, test 2021-2021
[TUNING] Rolling window: train 2016-2021, test 2022-2022

= BEST HYPER-PARAMETERS (based on average R2 across windows) =
Best config (n_layer

In [26]:
# from sklearn.model_selection import train_test_split
#
# train_df, test_df = train_test_split(data, test_size=0.2, random_state=1, shuffle=False)
#
# x_train = train_df[numeric_cols].values
# x_test  = test_df[numeric_cols].values
#
# y_train = train_df['ytm_change'].values
# y_test  = test_df['ytm_change'].values
#
#
# print("x_train mean/std:", np.mean(x_train), np.std(x_train))
# print("x_test  mean/std:", np.mean(x_test),  np.std(x_test))

In [27]:
# input_dim = x_train.shape[1]
# inputs = keras.Input(shape=(input_dim,))
#
# d = keras.layers.Dense(units=8, activation='relu')(inputs)
# d = keras.layers.Dense(units=8, activation='relu')(d)
# d = keras.layers.Dense(units=8, activation='relu')(d)
# outputs = keras.layers.Dense(units=1)(d)
#
# model1 = keras.Model(inputs=inputs, outputs=outputs)
# model1.summary()

In [28]:
# optimizer = keras.optimizers.Adam(
#     learning_rate=1e-4,
#     clipnorm=1.0
# )
#
# model1.compile(
#     loss='mse',
#     metrics=['mse', 'mae'],
#     optimizer=optimizer
# )
#
# early_stopping_cb = keras.callbacks.EarlyStopping(
#     patience=10,
#     restore_best_weights=True
# )
#
# history = model1.fit(
#     x=x_train,
#     y=y_train,
#     batch_size=64,
#     epochs=200,
#     shuffle=False,
#     validation_split=0.2,
#     callbacks=[early_stopping_cb],
#     verbose=2
# )


In [29]:
# from sklearn.metrics import r2_score
# from sklearn.metrics import median_absolute_error
#
# loss_test, mse_test, mae_test = model1.evaluate(x_test, y_test, verbose=0)
#
# y_pred_test = model1.predict(x_test, verbose=0).ravel()
#
# r2_test = r2_score(y_test, y_pred_test)
# medae_test = median_absolute_error(y_test, y_pred_test)
#
# print(medae_test)
# print(loss_test, mse_test, mae_test)
# print(r2_test)

In [30]:
# model1.summary()

## Classification

In [31]:
# eps = 0.001
#
# y_train2 = np.where(
#     y_train < -eps, 0,
#     np.where(y_train > eps, 2, 1)
# )
# y_test2 = np.where(
#     y_test < -eps, 0,
#     np.where(y_test > eps, 2, 1)
# )
#
# print("regression train size:", x_train.shape[0])
# print("classification train size:", y_train2.shape[0])

In [32]:
# print("x_train:", type(x_train), x_train.shape)
# print("y_train2:", type(y_train2), y_train2.shape, y_train2.dtype)
#
# if hasattr(x_train, "columns"):
#     print("x_train columns:", len(x_train.columns))

In [33]:
# print(type(y_train2), y_train2.shape, y_train2.dtype)
# print("unique labels:", np.unique(y_train2)[:10])

In [34]:
# input_dim = x_train.shape[1]
# inputs2 = keras.Input(shape=(input_dim,))
#
# h = keras.layers.Dense(8, activation="relu")(inputs2)
# h = keras.layers.Dense(8, activation="relu")(h)
# h = keras.layers.Dense(3, activation="relu")(h)
#
# outputs2 = keras.layers.Dense(3, activation="softmax")(h)
#
# model2 = keras.Model(inputs=inputs2, outputs=outputs2)
# model2.summary()


In [35]:
# opti = keras.optimizers.Adam(
#     learning_rate=3e-4,
#     clipnorm=1.0
# )
#
# model2.compile(
#     loss="sparse_categorical_crossentropy",
#     optimizer=opti,
#     metrics=["accuracy"]
# )
#
# history2 = model2.fit(
#     x=x_train,
#     y=y_train2,
#     batch_size=512,
#     epochs=15,
#     shuffle=False,
#     verbose=1
# )


In [36]:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
#
# y_prob_test = model2.predict(x_test, verbose=0)
# y_pred_test2 = np.argmax(y_prob_test, axis=1)
#
# acc = accuracy_score(y_test2, y_pred_test2)
# prec = precision_score(y_test2, y_pred_test2, average="macro", zero_division=0)
# rec = recall_score(y_test2,  y_pred_test2, average="macro", zero_division=0)
#
# print("Test Accuracy:", acc)
# print("Test Precision (macro):", prec)
# print("Test Recall (macro):", rec)